# GNN for edge classification to link the tracksters

Here, we will create an Edge classification graph that will link the tracksters belonging to the same particle. We will have the following properties:
1. Nodes: Tracksters
2. Node features: Trackster features
3. True label: From Associations tree
4. Edge features: .....

## Opening the root file

In [1]:
import uproot
import awkward as ak
import numpy as np

file=uproot.open("flat_tree_for_Neutron.root")

tracksters=file["ticlDumper/CLUE3DHighTree"].arrays(library="ak")
simtracksters=file["ticlDumper/SimTrackstersTree"].arrays(library="ak")
associations=file["ticlDumper/associations"].arrays(library="ak")

In [2]:
tracksters

<Array [{run_: 1, ...}, ..., {run_: 1, ...}] type='20000 * {run_: uint32, l...'>

In [3]:
associations

<Array [{run_: 1, ...}, ..., {run_: 1, ...}] type='20000 * {run_: uint32, l...'>

In [4]:
test_event=tracksters[0]

In [5]:
test_event

<Record {run_: 1, luminosityBlock_: 1, ...} type='{run_: uint32, luminosity...'>

In [6]:
test_event.time

<Array [-99, -99, 11.8, -99, ..., -99, 15.1, -99, -99] type='31 * float32'>

## Extracting the graph features

### Getting the node features

The tracksters are the nodes and the node features are the characteristics of the tracksters

In [7]:
def get_node_features(event):
    feats=np.stack([
        event["time"],
        event["raw_energy"],
        event["raw_em_energy"],
        event["raw_pt"],
        event["barycenter_x"],
        event["barycenter_y"],
        event["barycenter_z"],
        event["barycenter_eta"],
        event["barycenter_phi"],
        event["EV1"],
        event["EV2"],
        event["EV3"],
        event["sigmaPCA1"],
        event["sigmaPCA2"],
        event["sigmaPCA3"]
    ],axis=1)
    
    return np.array(feats)

In [8]:
# Testing
test_event=tracksters[0]
node_features=get_node_features(test_event)
node_features.shape

(31, 15)

### Building the graph edges

In [9]:
from sklearn.neighbors import NearestNeighbors

def build_edges(event,k=8):
    x=np.array(event["barycenter_x"])
    y=np.array(event["barycenter_y"])
    z=np.array(event["barycenter_z"])
    coords=np.stack([x,y,z],axis=1)
    # Fit KNN-determine the KNN
    n_nodes=len(coords)
    k_eff=min(k,n_nodes-1)
    if n_nodes<2:
        return np.empty((2,0),dtype=int)
    knn=NearestNeighbors(n_neighbors=k_eff+1,algorithm='auto').fit(coords)
    dist,idx=knn.kneighbors(coords)
    #Creating edges
    edges=[]
    for i,nbrs in enumerate(idx):
        for j in nbrs[1:]:
            edges.append([i,j])
    edges=np.array(edges).T
    return edges

In [10]:
# Testing
test_event=tracksters[0]
node_features=get_node_features(test_event)
edges=build_edges(test_event)
#edges
edges.shape

(2, 248)

### Build Edge Features

In [11]:
def build_edge_features(event,edge_index):
    x=np.array(event["barycenter_x"])
    y=np.array(event["barycenter_y"])
    z=np.array(event["barycenter_z"])
    eta=np.array(event["barycenter_eta"])
    phi=np.array(event["barycenter_phi"])
    t=np.array(event["time"])
    E=np.array(event["raw_energy"])
    
    features=[]
    
    for i,j in edge_index.T:
        dx=x[i]-x[j]
        dy=y[i]-y[j]
        dz=z[i]-z[j]
        dist=np.sqrt(dx*dx+dy*dy+dz*dz)
        deta=eta[i]-eta[j]
        dphi=phi[i]-phi[j]
        dt=t[i]-t[j]
        dE=E[i]-E[j]
        features.append([dist,dz,deta,dphi,dt,dE])
    return np.array(features)

In [12]:
# Testing
test_event=tracksters[0]
node_features=get_node_features(test_event)
edges=build_edges(test_event)
#edges
edge_features=build_edge_features(test_event,edges)
edge_features.shape

(248, 6)

### Getting the truth mappings

In [13]:
def get_truth_mappings(event_id):
    reco_to_sim=associations[event_id]["CLUE3DHigh_recoToSim_CP"]
    truth={}
    for i,match in enumerate(reco_to_sim):
        if len(match)>0:
            truth[i]=int(match[0])
    return truth

In [14]:
#Testing
get_truth_mappings(0)

{0: 0,
 1: 0,
 2: 0,
 3: 0,
 4: 0,
 5: 0,
 6: 0,
 7: 0,
 8: 0,
 9: 0,
 10: 0,
 11: 0,
 12: 0,
 13: 0,
 14: 0,
 15: 0,
 16: 0,
 17: 0,
 18: 0,
 19: 0,
 20: 0,
 21: 0,
 22: 0,
 23: 0,
 24: 0,
 25: 0,
 26: 0,
 27: 0,
 28: 0,
 29: 0,
 30: 0}

### Building edge labels

In [15]:
def build_edge_labels(edge_index,truth):
    labels=[]
    for i,j in edge_index.T:
        if i in truth and j in truth and truth[i]==truth[j]:
            labels.append(1)
        else:
            labels.append(0)
    return np.array(labels)

In [16]:
# Testing
event_id=0
test_event=tracksters[event_id]
node_features=get_node_features(test_event)
edge_index=build_edges(test_event)
#edges
edge_features=build_edge_features(test_event,edge_index)
#truth
truth=get_truth_mappings(event_id)
edge_labels=build_edge_labels(edge_index,truth)
edge_labels

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1])

## Building the graph object

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data,Dataset
from torch_geometric.loader import DataLoader

In [18]:
def build_graph(event_id):
    event=tracksters[event_id]
    node_features=get_node_features(event)
    edge_index=build_edges(event)
    edge_attr=build_edge_features(event,edge_index)
    truth=get_truth_mappings(event_id)
    labels=build_edge_labels(edge_index,truth)
    data=Data(x=torch.tensor(node_features),
              edge_index=torch.tensor(edge_index),
              edge_attr=torch.tensor(edge_attr),
              y=torch.tensor(labels))
    return data

In [19]:
def build_graph(event_id):
    event=tracksters[event_id]
    node_features=get_node_features(event)
    edge_index=build_edges(event)
    edge_attr=build_edge_features(event,edge_index)
    truth=get_truth_mappings(event_id)
    labels=build_edge_labels(edge_index,truth)
    data=Data(x=torch.tensor(node_features,dtype=torch.float),
              edge_index=torch.tensor(edge_index,dtype=torch.long),
              edge_attr=torch.tensor(edge_attr,dtype=torch.float)
             )
    data.edge_label=torch.tensor(labels,dtype=torch.float)
    return data

In [20]:
#Testing- for the first event
event_idx=0
test_graph=build_graph(event_id=event_idx)
test_graph

Data(x=[31, 15], edge_index=[2, 248], edge_attr=[248, 6], edge_label=[248])

In [21]:
#Testing- for the first event
event_idx=3
graph=build_graph(event_id=event_idx)
graph

Data(x=[59, 15], edge_index=[2, 472], edge_attr=[472, 6], edge_label=[472])

In [36]:
graph.edge_label

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 

## Building the dataset

In [22]:
#%%time
#graphs=[]
#for i in range(20000):
#    graphs.append(build_graph(event_id=i))

In [23]:
%%time
graphs=[]
for i in range(1000):
    graphs.append(build_graph(event_id=i))

CPU times: total: 2min 11s
Wall time: 22.2 s


In [24]:
#%time
#graphs

## Splitting the Dataset into training, testing and validation

In [25]:
from sklearn.model_selection import train_test_split
#First split
train_data,temp_data=train_test_split(graphs,test_size=0.3,random_state=42)
#Second split
val_data,test_data=train_test_split(temp_data,test_size=0.7,random_state=42)
#Printing the dataset sizes
print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Testing samples: {len(test_data)}")

Training samples: 700
Validation samples: 90
Testing samples: 210


## Using DataLoader

In [26]:
from torch_geometric.loader import DataLoader
batch_size=32
train_loader=DataLoader(train_data,batch_size=batch_size,shuffle=True)
val_loader=DataLoader(val_data,batch_size=batch_size,shuffle=True)
test_loader=DataLoader(test_data,batch_size=batch_size,shuffle=True)

## Building the Edge Classification GCN

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class Edge_Classifier(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        super().__init__()
        #First GCN Layer
        self.conv1=GCNConv(input_dim,hidden_dim)
        #Stacking 6 Repeated blocks
        self.convs=nn.ModuleList()
        for i in range(6):
            self.convs.append(GCNConv(hidden_dim,hidden_dim))
        #Final layer
        self.final_conv=GCNConv(hidden_dim,hidden_dim)
        #Regularization
        self.dropout=nn.Dropout(p=0.1)
        #The Edge Classification MLP
        self.edge_mlp=nn.Sequential(
            nn.Linear(2*hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,1)
        )
    def forward(self,data):
        x,edge_index,batch=data.x,data.edge_index,data.batch
        #-----------Passing the GCN forward passes--------------#
        ##### First GCN ######
        x=self.conv1(x,edge_index)
        x=F.relu(x)
        x=self.dropout(x)
        ##### Six repeated convolutions #####
        for conv in self.convs:
            x=conv(x,edge_index)
            x=F.relu(x)
            x=self.dropout(x)
        ##### Final convolution #####
        x=self.final_conv(x,edge_index)
        node_embedding=x
        #------------ Edge Classification MLP -----------------#
        row,col=edge_index
        edge_features=torch.cat([x[row],x[col]],dim=1)
        edge_logits=self.edge_mlp(edge_features).view(-1)
        return edge_logits

## Setting up the device and model

In [28]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [29]:
node_dim=graphs[0].x.shape[1]
model=Edge_Classifier(input_dim=node_dim,hidden_dim=64).to(device)

In [30]:
model

Edge_Classifier(
  (conv1): GCNConv(15, 64)
  (convs): ModuleList(
    (0-5): 6 x GCNConv(64, 64)
  )
  (final_conv): GCNConv(64, 64)
  (dropout): Dropout(p=0.1, inplace=False)
  (edge_mlp): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [31]:
model.parameters()

<generator object Module.parameters at 0x00000250EBDB7E60>

## Training, Validation and Testing

In [32]:
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
criterion=nn.BCEWithLogitsLoss()

### Training

In [35]:
# Traiing loop
def train():
    model.train()
    total_loss=0.0
    for batch in train_loader:
        batch=batch.to(device)
        optimizer.zero_grad()
        #Forward pass
        out=model(batch)
        #Compute Loss
        loss=criterion(out,batch.edge_label)
        #Backpropagation
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(train_loader)